# 13 · Blind carrier-phase recovery

After frequency-offset estimation, the laser phase noise
([09](09-laser-phase-noise.ipynb)) remains: a slowly wandering rotation of the
constellation.  Carrier-phase recovery (CPR) estimates it from the received
symbols and removes it.  *Blind* methods use no known data; they work on
the symbol-rate output of the matched filter or the equalizer.

1. Viterbi-Viterbi (M-th power) and blind phase search (BPS);
2. the window trade-off: noise averaging against tracking;
3. MAP estimation with a phase-noise prior (Tikhonov);
4. the rotational **ambiguity** and how it is resolved;
5. **cycle slips** and their repair;
6. joint estimation over two polarizations.

The API is the same for every method: `estimate_carrier_phase(y, method)`
returns a `CarrierPhaseEstimate`; `correct_carrier_phase(y, method)`
estimates and removes in one call.  Pilot-based methods follow in
[14](14-pilot-aided-carrier-recovery.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device
from commkit.recovery import BPS, CycleSlip, Tikhonov, ViterbiViterbi

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
RS = 32e9
N = 2**16


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(C16, num_symbols=N, symbol_rate=RS, rng=1, device=DEVICE)


def received(linewidth, esn0_db=16.0, rng=2):
    """Symbols with a known phase-noise trajectory and AWGN; returns (signal, phase)."""
    phi = ck.impairments.generate_phase_noise(
        num_samples=N, sampling_rate=RS, linewidth=linewidth, rng=rng, device=DEVICE
    )
    rot = tx.replace(
        samples=(tx.samples * tx.xp.exp(1j * phi)).astype(tx.samples.dtype)
    )
    return ck.impairments.apply_awgn(rot, esn0_db=esn0_db, rng=rng + 1), host(phi)


def score(y):
    """SNR after resolving the 90-degree ambiguity against the reference."""
    return ck.metrics.snr(ck.recovery.resolve_phase_ambiguity(y))

## 1. Viterbi-Viterbi and blind phase search

* **Viterbi-Viterbi** (`ViterbiViterbi`) raises the symbols to the 4th
  power, which removes QPSK modulation entirely, averages over a block and
  takes a quarter of the angle.  For 16-QAM the 4th power leaves a large
  "self-noise", because the inner and outer points do not collapse to one
  phase, so it needs long blocks (128 here) and still slips now and then.
* **Blind phase search** (`BPS`) tries `test_phases` candidate rotations in
  each block, decides the symbols for each, and keeps the rotation with the
  smallest decision error.  It works for any QAM and is the standard for
  16-QAM and beyond, at the cost of `test_phases` decisions per symbol.

A 1 MHz combined linewidth at 32 GBd and Es/N0 = 16 dB:

In [ ]:
rx, phi = received(1e6)
for method in (ViterbiViterbi(block_size=128), BPS()):
    est = ck.recovery.estimate_carrier_phase(rx, method)
    y = ck.recovery.correct_carrier_phase(rx, est)
    err = host(est.value) - phi
    err -= (
        np.round(np.median(err) / (np.pi / 2)) * np.pi / 2
    )  # remove the multiple of 90 degrees
    print(
        f"{type(method).__name__:15s} RMS phase error {np.sqrt(np.mean(err**2)):.3f} rad, SNR {score(y):5.2f} dB"
    )
ck.plotting.plot_carrier_phase_trajectory(
    est.value, title="BPS estimate (1 MHz linewidth)", show=True
)

## 2. The window trade-off

Each method averages over a block of symbols (`block_size`).  A longer
block averages more noise away, but the phase must stay nearly constant
inside it.  Both ends fail abruptly rather than gracefully: too short a
block lets the noise push single estimates past 45°, too long a block
cannot follow the phase, and either way the trajectory **slips** by 90°
(section 5) and the SNR collapses.  The usable range shrinks as `Δν·T`
grows.

In [ ]:
blocks = [8, 16, 32, 64, 128, 256, 512]
fig, ax = plt.subplots(figsize=(8, 4))
for lw in (100e3, 1e6, 3e6):
    r, _ = received(lw, rng=4)
    ax.semilogx(
        blocks,
        [
            score(ck.recovery.correct_carrier_phase(r, BPS(block_size=b)))
            for b in blocks
        ],
        "o-",
        label=f"Δν = {lw / 1e6:g} MHz (Δν·T = {lw / RS:.0e})",
    )
ax.axhline(16, color="gray", ls=":", label="Es/N0")
ax.set_ylim(-10, 18)
ax.set_xlabel("BPS block size [symbols]")
ax.set_ylabel("SNR after CPR [dB]")
ax.legend()
plt.show()

## 3. MAP estimation: Tikhonov

If the linewidth and the SNR are known, the best estimate of a Wiener
phase is a **Kalman/Rauch-Tung-Striebel smoother**: it weights each block's
measurement against the random-walk prior, forward and backward in time,
instead of trusting one fixed window.
`Tikhonov(linewidth=Δν, snr_db=...)` does this on 4th-power block
measurements, taking the symbol period from the Signal.  Its home is a low SNR, where the fixed window of
Viterbi-Viterbi is either too noisy or too slow.  QPSK at Es/N0 = 8 dB and
a 5 MHz linewidth:

In [ ]:
qpsk = ck.generate(
    Constellation.psk(4), num_symbols=N, symbol_rate=RS, rng=20, device=DEVICE
)
phi_q = ck.impairments.generate_phase_noise(
    num_samples=N, sampling_rate=RS, linewidth=5e6, rng=21, device=DEVICE
)
r_q = ck.impairments.apply_awgn(
    qpsk.replace(
        samples=(qpsk.samples * qpsk.xp.exp(1j * phi_q)).astype(qpsk.samples.dtype)
    ),
    esn0_db=8,
    rng=22,
)
for method in (
    ViterbiViterbi(block_size=16),
    ViterbiViterbi(block_size=64),
    ViterbiViterbi(block_size=256),
    Tikhonov(linewidth=5e6, snr_db=8),
):
    est = ck.recovery.estimate_carrier_phase(r_q, method)
    err = host(est.value) - host(phi_q)
    err -= np.round(np.median(err) / (np.pi / 2)) * np.pi / 2
    y = ck.recovery.resolve_phase_ambiguity(ck.recovery.correct_carrier_phase(r_q, est))
    print(
        f"{type(method).__name__:15s} block {method.block_size:3d}: RMS phase error {np.sqrt(np.mean(err**2)):.3f} rad, "
        f"BER {ck.metrics.ber(y):.2e}"
    )

## 4. The rotational ambiguity

A blind estimator sees the constellation, not the data, so it cannot tell a
rotation by 90° from none ([01](01-constellations-and-bit-mapping.ipynb) · section 4).  Its output is
correct up to a multiple of 90°, and the BER of a wrong multiple is about
one half.  Two ways out:

* **known symbols**: `resolve_phase_ambiguity` tries the four rotations
  against the reference (a few known symbols suffice in practice);
* **differential coding**: encode the bits in the *change* of quadrant, so
  a constant rotation does not matter (at about twice the BER).

In [ ]:
# The same link with the carrier starting at 100 degrees instead of 0.
start = rx.replace(samples=rx.samples * np.complex64(np.exp(1j * np.deg2rad(100))))
y = ck.recovery.correct_carrier_phase(start, BPS())
print(f"BER after BPS:                       {ck.metrics.ber(y):.3f}")
print(
    f"BER after resolve_phase_ambiguity:   {ck.metrics.ber(ck.recovery.resolve_phase_ambiguity(y)):.2e}"
)

## 5. Cycle slips

When the noise briefly pushes the estimate past 45°, the unwrapped
trajectory jumps to the neighbouring branch: a **cycle slip**.  Everything
after it is rotated by 90°, which a single ambiguity resolution cannot fix:
the BER jumps to 1/2 from that point on.  Short blocks and high linewidth
make slips likely.

`CycleSlip()`, nested in a method, predicts each block's phase by a line
through the last `history` blocks and snaps a deviation beyond `threshold`
(45°) back by 90°.  The line lags the wandering phase, and one false
detection leaves a lasting slip, so the phase must not wander far over the
history: keep `2π Δν T · history · block_size < threshold² / 2` (0.31 rad²
at 45°), or the repair "fixes" genuine phase noise and causes the slips it
should remove.  By default `history` spans 640 symbols whatever the block
size, which holds up to about 2.4 MHz at 32 GBd.  Here: 80 blocks of 8
symbols at 1 MHz, 0.13 rad².  (Pilots, next notebook, catch slips for
certain.)

In [ ]:
r, phi = received(1e6, rng=8)
fig, ax = plt.subplots(figsize=(8, 4))
for method in (BPS(block_size=8), BPS(block_size=8, cycle_slip=CycleSlip())):
    est = ck.recovery.estimate_carrier_phase(r, method)
    err = host(est.value) - phi
    ax.plot(
        err / (np.pi / 2),
        label=f"{'with' if method.cycle_slip else 'without'} CycleSlip, "
        f"SNR {score(ck.recovery.correct_carrier_phase(r, est)):.1f} dB",
    )
ax.set_xlabel("Symbol")
ax.set_ylabel("Estimate - truth [multiples of 90°]")
ax.set_title("BPS with 8-symbol blocks, 1 MHz linewidth")
ax.legend()
plt.show()

## 6. Two polarizations, one LO

With a shared LO both polarizations carry the same phase noise
([09](09-laser-phase-noise.ipynb) · section 4).  `joint_channels=True` estimates one phase from
both, doubling the symbols per block, which allows shorter blocks for the
same noise averaging.  At 2 MHz and Es/N0 = 14 dB, 16-symbol blocks slip
on each polarization alone and hold when pooled:

In [ ]:
dual = ck.generate(
    C16, num_symbols=N, symbol_rate=RS, num_channels=2, rng=10, device=DEVICE
)
r2 = ck.impairments.apply_awgn(
    ck.impairments.apply_phase_noise(dual, linewidth=2e6, shared_lo=True, rng=11),
    esn0_db=14,
    rng=12,
)
for joint in (False, True):
    y = ck.recovery.correct_carrier_phase(r2, BPS(block_size=16, joint_channels=joint))
    print(
        f"joint_channels={joint!s:5}: SNR per polarization {np.round(score(y), 2)} dB"
    )

**Summary**

* BPS is the general-purpose blind CPR for QAM; the 4th-power methods
  (Viterbi-Viterbi, Tikhonov) suit QPSK, where Tikhonov's smoother beats
  any fixed window.
* `block_size` trades noise averaging against tracking; outside its usable
  range the estimate slips, and that range shrinks with `Δν·T`.
* Blind CPR leaves a 90° ambiguity (resolve it with known symbols) and can
  slip (repair with `CycleSlip()` sized to the linewidth, or use pilots).
* `joint_channels=True` pools both polarizations of a shared LO.

**Next:** [14 · Pilot-aided carrier recovery](14-pilot-aided-carrier-recovery.ipynb).